# Modelica to PyPowSyBl: Dynamic Simulation Toolkit

### Industrialized Workflow
1. **Create Network:** Programmatically build the static loadflow structure via PyPowSyBl.
2. **Initialize Loadflow:** Define load and generation profiles.
3. **Run Loadflow:** Execute the base AC Loadflow using OpenLF.
4. **Dynamic Models Setup:** Map precompiled models (e.g., `GeneratorPV`) directly from the Dynawo database.
5. **Physical & Initialization Parameters:** Build the `.PAR` files using physical parameters and Loadflow initialization references.
6. **Execute Dynamic Simulation:** Run Dynawo time-domain simulation for events.

## Step 0: Environment Setup (Dynawo Install & Config)

Before running dynamic simulations, we need the Dynawo engine installed and configured so PyPowSyBl can find it.

In [ ]:
# --- 1. CHECK DYNAWO INSTALLATION ---
import os
import re
import datetime

config_dir = os.path.expanduser("~/.itools")
config_file = os.path.join(config_dir, "config.yml")
dynawo_sys_path = None

# Check if config.yml exists and read its content
if os.path.exists(config_file):
    with open(config_file, "r") as f:
        content = f.read()
        # Search for the homeDir value under the dynawo section using regex
        match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', content)
        if match:
            dynawo_sys_path = match.group(1)

# Verify if a path was extracted and if it actually exists on your computer
if not dynawo_sys_path or not os.path.exists(dynawo_sys_path):
    raise FileNotFoundError(
        f"Error! No valid Dynawo installation found on the system. "
        f"Detected path: '{dynawo_sys_path}'. Make sure Dynawo is installed and properly configured in {config_file}."
    )

print(f"Configuration correct! Dynawo installation validated at: {dynawo_sys_path}")

# Save the verified path in an environment variable to use in the next cell
os.environ["DYNAWO_SYS_PATH"] = dynawo_sys_path

In [ ]:
# --- 2. CONFIGURE ENVIRONMENT TEMPLATE ---

config_dir = os.path.expanduser("~/.itools")
target_config = os.path.join(config_dir, "config.yml")
template_config = "./getting_started_data/config.yml"
current_dir = os.getcwd()
dynawo_sys_path = os.environ.get("DYNAWO_SYS_PATH")

# Ensure the previous cell was executed successfully
if not dynawo_sys_path:
    raise ValueError("The Dynawo path is not defined. Please run the previous cell first.")

if os.path.exists(target_config):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(config_dir, f"config_{timestamp}.yml")
    os.rename(target_config, backup_path)
    print(f"Backup created at: {backup_path}")

with open(template_config, "r") as f:
    template_content = f.read()

template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_sys_path}"')
template_content = template_content.replace("WORKING_DIR", current_dir)

os.makedirs(config_dir, exist_ok=True)
with open(target_config, "w") as f:
    f.write(template_content)

print(f"\nDone! Template successfully configured and saved to {target_config}:\n")
print(template_content)

In [ ]:
# --- 3. IMPORT LIBRARIES AND BASE CONFIGURATION ---
import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import matplotlib.pyplot as plt
from IPython.display import display

# Ensure plots are displayed inline within the Jupyter environment
%matplotlib inline

print(f"PyPowSyBl version: {pp.__version__}")

current_dir = os.getcwd()
dynawo_dir = os.path.join(current_dir, "dynawo")
data_dir = os.path.join(current_dir, "getting_started_data")

## Step 1 & 2: Create Network and Initialize Loadflow Profiles

We construct a highly meshed 12-Bus ring network programmatically. This features 10 generators (`GEN_1` acting as the Slack bus), 5 loads distributed across the grid, and 2 linear shunt compensators to support voltage profiles.

In [ ]:
# --- 4. CREATE NETWORK AND INITIALIZE PROFILES ---
def create_complex_ring_network() -> pp.network.Network:
    """
    Constructs a 12-bus ring static network with multiple generators, loads, and shunts.
    """
    net = pp.network.load("getting_started_data/Base_Case.xiidm")

    # Create Core Topology (12 Substations, Voltage Levels, and Buses)
    bus_ids = [f"BUS_{i}" for i in range(1, 13)]
    vl_ids = [f"VL_{i}" for i in range(1, 13)]
    sub_ids = [f"SUB_{i}" for i in range(1, 13)]

    net.create_substations(id=sub_ids, name=[f"Substation {i}" for i in range(1, 13)])
    net.create_voltage_levels(
        id=vl_ids,
        name=vl_ids,
        substation_id=sub_ids,
        topology_kind=["BUS_BREAKER"] * 12,
        nominal_v=[400.0] * 12,
    )
    net.create_buses(id=bus_ids, name=bus_ids, voltage_level_id=vl_ids)

    # Add 10 Generators (BUS_1 to BUS_10). GEN_1 acts as Slack
    gen_ids = [f"GEN_{i}" for i in range(1, 11)]
    net.create_generators(
        id=gen_ids,
        name=gen_ids,
        voltage_level_id=vl_ids[:10],
        bus_id=bus_ids[:10],
        target_p=[250.0 if i == 1 else 100.0 for i in range(1, 11)],
        target_v=[400.0] * 10,
        min_p=[0.0] * 10,
        max_p=[500.0] * 10,
        voltage_regulator_on=[True] * 10,
    )

    # Add 5 Distributed Loads
    load_nodes = [3, 5, 7, 9, 11]
    load_ids = [f"LOAD_{i}" for i in load_nodes]
    net.create_loads(
        id=load_ids,
        name=load_ids,
        voltage_level_id=[f"VL_{i}" for i in load_nodes],
        bus_id=[f"BUS_{i}" for i in load_nodes],
        p0=[180.0] * 5,
        q0=[60.0] * 5,
    )

    # Add 2 Linear Shunt Compensators for reactive support
    shunt_df = pd.DataFrame(
        {
            "id": ["SHUNT_4", "SHUNT_8"],
            "voltage_level_id": ["VL_4", "VL_8"],
            "bus_id": ["BUS_4", "BUS_8"],
            "model_type": ["LINEAR", "LINEAR"],
            "section_count": [1, 1],
        }
    ).set_index("id")

    linear_model_df = pd.DataFrame(
        {
            "id": ["SHUNT_4", "SHUNT_8"],
            "g_per_section": [0.0, 0.0],
            "b_per_section": [0.0008, 0.0008],
            "max_section_count": [1, 1],
        }
    ).set_index("id")

    net.create_shunt_compensators(shunt_df, linear_model_df)

    # Connect the buses in a ring structure
    line_ids = [f"LINE_{i}_{i + 1}" for i in range(1, 12)] + ["LINE_12_1"]
    b1_ids = [f"BUS_{i}" for i in range(1, 12)] + ["BUS_12"]
    b2_ids = [f"BUS_{i + 1}" for i in range(1, 12)] + ["BUS_1"]
    vl1_ids = [f"VL_{i}" for i in range(1, 12)] + ["VL_12"]
    vl2_ids = [f"VL_{i + 1}" for i in range(1, 12)] + ["VL_1"]

    net.create_lines(
        id=line_ids,
        name=line_ids,
        voltage_level1_id=vl1_ids,
        bus1_id=b1_ids,
        voltage_level2_id=vl2_ids,
        bus2_id=b2_ids,
        r=[1.5] * 12,
        x=[15.0] * 12,
        b1=[0.0005] * 12,
        b2=[0.0005] * 12,
    )

    return net


print("Creating 12-Bus complex network...")
network = create_complex_ring_network()

## Step 3: Run Loadflow

Execute the base Loadflow using PyPowSyBl's OpenLF. The engine calculates the steady-state across the ring topology, resolving the active/reactive power flow and the final voltage profile for all 12 buses.

In [ ]:
# --- 5. RUN STEADY-STATE LOADFLOW ---
print("Running Loadflow (OpenLF)...")
lf_parameters = pp.loadflow.Parameters()
lf_results = pp.loadflow.run_ac(network, parameters=lf_parameters)

print(f"Loadflow Status: {lf_results[0].status.name}")
display(network.get_buses()[["name", "v_mag", "v_angle"]])

## Step 4: Dynamic Models Setup

Map the 10 static generators to their precompiled dynamic representations. We assign the highly detailed `GeneratorSynchronousThreeWindingsGoverNordicVRNordic` class to all 10 machines sequentially using a Pandas DataFrame to optimize the code.

In [ ]:
# --- 6. CONFIGURE DYNAMIC MODELS MAPPING ---
print("Mapping static elements to precompiled dynamic Modelica classes...")
mapping = dyn.ModelMapping()

# Bulk mapping for 10 Generators
df_gen = pd.DataFrame(
    [
        {
            "static_id": f"GEN_{i}",
            "parameter_set_id": f"GEN_{i}",
            "model_name": "GeneratorSynchronousThreeWindingsGoverNordicVRNordic",
        }
        for i in range(1, 11)
    ]
).set_index("static_id", drop=False)

mapping.add_synchronous_generator(df_gen)
print("Mapping complete for 10 generators.")

## Step 5: Physical Parameters and Loadflow Initialization

Construct the `.PAR` files. To avoid dealing with hundreds of lines of static strings, we dynamically build the XML configuration loop, mapping the $P$, $Q$, $V$, and $\theta$ IIDM references automatically for all 10 generator instances.

In [ ]:
# --- 7. GENERATE PHYSICAL PARAMETERS (PAR FILES) ---
print("Generating dynamic physical parameters and LF initialization mapping...")

# Constructing the Generator XML string programmatically
xml_dynamic_model_par = """<?xml version="1.0" encoding="UTF-8"?>\n<parametersSet xmlns="http://www.rte-france.com/dynawo">\n"""

gen_template = """    <set id="{gen_id}">
        <par type="DOUBLE" name="generator_DPu" value="0"/>
        <par type="INT" name="generator_ExcitationPu" value="1"/>
        <par type="DOUBLE" name="generator_H" value="3"/>
        <par type="DOUBLE" name="generator_MdPuEfd" value="1"/>
        <par type="DOUBLE" name="generator_PNomAlt" value="4275"/>
        <par type="DOUBLE" name="generator_PNomTurb" value="4275"/>
        <par type="DOUBLE" name="generator_RTfPu" value="0"/>
        <par type="DOUBLE" name="generator_RaPu" value="0.002"/>
        <par type="DOUBLE" name="generator_SNom" value="4500"/>
        <par type="DOUBLE" name="generator_SnTfo" value="4500"/>
        <par type="DOUBLE" name="generator_Tpd0" value="5"/>
        <par type="DOUBLE" name="generator_Tppd0" value="0.05"/>
        <par type="DOUBLE" name="generator_Tppq0" value="0.1"/>
        <par type="DOUBLE" name="generator_UBaseHV" value="15"/>
        <par type="DOUBLE" name="generator_UBaseLV" value="15"/>
        <par type="DOUBLE" name="generator_UNom" value="15"/>
        <par type="DOUBLE" name="generator_UNomHV" value="15"/>
        <par type="DOUBLE" name="generator_UNomLV" value="15"/>
        <par type="DOUBLE" name="generator_XTfPu" value="0"/>
        <par type="DOUBLE" name="generator_XdPu" value="1.1"/>
        <par type="DOUBLE" name="generator_XlPu" value="0.15"/>
        <par type="DOUBLE" name="generator_XpdPu" value="0.25"/>
        <par type="DOUBLE" name="generator_XppdPu" value="0.2"/>
        <par type="DOUBLE" name="generator_XppqPu" value="0.2"/>
        <par type="DOUBLE" name="generator_XqPu" value="0.7"/>
        <par type="DOUBLE" name="generator_md" value="0.1"/>
        <par type="DOUBLE" name="generator_mq" value="0.1"/>
        <par type="DOUBLE" name="generator_nd" value="6.0257"/>
        <par type="DOUBLE" name="generator_nq" value="6.0257"/>
        <par type="DOUBLE" name="governor_KSigma" value="0.08"/>
        <par type="DOUBLE" name="governor_Ki" value="0.4"/>
        <par type="DOUBLE" name="governor_Kp" value="2"/>
        <par type="DOUBLE" name="governor_PNom" value="4275"/>
        <par type="DOUBLE" name="voltageRegulator_EfdMaxPu" value="4"/>
        <par type="DOUBLE" name="voltageRegulator_IrLimPu" value="1.8991"/>
        <par type="DOUBLE" name="voltageRegulator_KPss" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_KTgr" value="70"/>
        <par type="DOUBLE" name="voltageRegulator_OelMode" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_tDerOmega" value="1"/>
        <par type="DOUBLE" name="voltageRegulator_tLagPss" value="0.01"/>
        <par type="DOUBLE" name="voltageRegulator_tLagTgr" value="20"/>
        <par type="DOUBLE" name="voltageRegulator_tLeadPss" value="1"/>
        <par type="DOUBLE" name="voltageRegulator_tLeadTgr" value="10"/>
        <par type="DOUBLE" name="voltageRegulator_tOelMin" value="-11"/>
        <par type="BOOL" name="generator_UseApproximation" value="true"/>
        <reference type="DOUBLE" name="generator_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="generator_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="generator_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="generator_UPhase0" origData="IIDM" origName="angle"/>
    </set>
"""

for i in range(1, 11):
    xml_dynamic_model_par += gen_template.format(gen_id=f"GEN_{i}")

xml_dynamic_model_par += "</parametersSet>\n"

xml_static_network_par = """<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="Network">
        <par type="DOUBLE" name="capacitor_no_reclosing_delay" value="300.0"/>
        <par type="DOUBLE" name="dangling_line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="load_Tp" value="90.0"/>
        <par type="DOUBLE" name="load_Tq" value="90.0"/>
        <par type="DOUBLE" name="load_alpha" value="1.0"/>
        <par type="DOUBLE" name="load_alphaLong" value="0.0"/>
        <par type="DOUBLE" name="load_beta" value="2.0"/>
        <par type="DOUBLE" name="load_betaLong" value="0.0"/>
        <par type="BOOL" name="load_isControllable" value="false"/>
        <par type="BOOL" name="load_isRestorative" value="false"/>
        <par type="DOUBLE" name="load_zPMax" value="100.0"/>
        <par type="DOUBLE" name="load_zQMax" value="100.0"/>
        <par type="DOUBLE" name="reactance_no_reclosing_delay" value="0.0"/>
        <par type="DOUBLE" name="transformer_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="transformer_t1st_HT" value="30.0"/>
        <par type="DOUBLE" name="transformer_t1st_THT" value="30.0"/>
        <par type="DOUBLE" name="transformer_tNext_HT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tNext_THT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tolV" value="0.0149999997"/>
        <par type="BOOL" name="BUS_12_hasShortCircuitCapabilities" value="true"/>
    </set>
</parametersSet>
"""

data_dir = "getting_started_data"
os.makedirs(data_dir, exist_ok=True)
with open(os.path.join(data_dir, "Base_Case.par"), "w") as f:
    f.write(xml_dynamic_model_par)
with open(os.path.join(data_dir, "Network.par"), "w") as f:
    f.write(xml_static_network_par)
print("Parameters saved and dynamically scaled.")

## Step 6: Execute Dynamic Simulation

Introduce a profound temporal fault (short-circuit) at `BUS_12` (the end of the ring) and track how the voltage propagates and recovers across multiple generators distributed throughout the network.

In [ ]:
# --- 8. EXECUTE DYNAMIC SIMULATION AND PLOT ---
print("Configuring short-circuit event on BUS_12...")
events = dyn.EventMapping()
events.add_node_fault(static_id="BUS_12", start_time=2.0, fault_time=0.1, r_pu=0.0, x_pu=0.0001)

outputs = dyn.OutputVariableMapping()
outputs.add_standard_model_curves("BUS_1", "U_value")  # Slack Generator
outputs.add_standard_model_curves("BUS_6", "U_value")  # Mid-ring Generator
outputs.add_standard_model_curves("BUS_12", "U_value")  # Fault location

sim_parameters = dyn.Parameters(start_time=0.0, stop_time=10.0)
simulation = dyn.Simulation()

print("Running dynamic simulation on the 12-Bus System...")
results = simulation.run(
    network,
    model_mapping=mapping,
    event_mapping=events,
    timeseries_mapping=outputs,
    parameters=sim_parameters,
)

if results.status().name == "SUCCESS":
    print("Simulation Successful!")
    curves = results.curves()
    plt.figure(figsize=(12, 6))
    plt.plot(
        curves.index,
        curves["NETWORK_BUS_12_U_value"],
        label="Bus 12 (Fault Location)",
        linewidth=2,
    )
    plt.plot(
        curves.index,
        curves["NETWORK_BUS_1_U_value"],
        label="Bus 1 (Slack Generator)",
        linestyle="--",
    )
    plt.plot(
        curves.index,
        curves["NETWORK_BUS_6_U_value"],
        label="Bus 6 (Mid-Ring Generator)",
        linestyle=":",
    )
    plt.title("Transient Response: Propagation of Fault in 12-Bus Ring Network")
    plt.xlabel("Time (s)")
    plt.ylabel("Voltage (PU)")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print(f"Simulation failed: {results.status_text()}")